# 📊 Reconciliation Data Extractor
### Automates extraction from 400+ Excel reconciliation templates

**Template Structure:**
- `Column C` → Reconciliation Element (rows 7–41)
- `Column D` → Original Data (rows 7–41)
- `Column G` → Discrepancies Details (rows 7–41)
- `Column I` → Evidence (rows 7–41)

**Outputs:**
1. `consolidated_output.xlsx` — all records from all files
2. `consolidated_output.csv` — same, as CSV
3. `field_level/` folder — one `.xlsx` per reconciliation field (~35 files)


## Cell 1 — Install & Import Libraries

In [ ]:
# Install required libraries (run once; skip if already installed)
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for lib in ["pandas", "openpyxl"]:
    install(lib)

print("✅ Libraries ready.")

In [ ]:
import os
import re
import logging
import traceback
from pathlib import Path

import pandas as pd
import openpyxl

print(f"pandas  : {pd.__version__}")
print(f"openpyxl: {openpyxl.__version__}")
print("✅ All imports successful.")

## Cell 2 — Configuration: Input / Output Paths

In [ ]:
# ─────────────────────────────────────────────────────────────────
# ✏️  EDIT THESE PATHS before running
# ─────────────────────────────────────────────────────────────────

# Folder that contains all 400 reconciliation Excel files
INPUT_FOLDER = r"C:\reconciliation_templates"          # ← change this

# Where consolidated outputs will be saved
OUTPUT_FOLDER = r"C:\reconciliation_output"            # ← change this

# Sub-folder for field-level Excel files
FIELD_OUTPUT_FOLDER = os.path.join(OUTPUT_FOLDER, "field_level")

# ─────────────────────────────────────────────────────────────────
# Sheet identification — tries these names in order; uses first match
# Update to match the actual sheet tab name(s) in your templates
# ─────────────────────────────────────────────────────────────────
TARGET_SHEET_NAMES = [
    "Inventory Recon & Evidence",
    "Inventory Recon",
    "Recon",
    "Sheet1",          # fallback
]

# Excel row range to extract (1-based, inclusive)
ROW_START = 7
ROW_END   = 41

# Column indices (1-based: A=1, B=2, C=3 …)
COL_RECON_ELEMENT   = 3   # C
COL_ORIGINAL_DATA   = 4   # D
COL_DISCREPANCIES   = 7   # G
COL_EVIDENCE        = 9   # I

# Log file location
LOG_FILE = os.path.join(OUTPUT_FOLDER, "extraction_log.txt")

# ─────────────────────────────────────────────────────────────────
# Create output folders
# ─────────────────────────────────────────────────────────────────
Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)
Path(FIELD_OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

print(f"Input  folder : {INPUT_FOLDER}")
print(f"Output folder : {OUTPUT_FOLDER}")
print(f"Field  folder : {FIELD_OUTPUT_FOLDER}")

## Cell 3 — Logging Setup

In [ ]:
# Configure logging: writes to both console and a log file
Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ]
)
logger = logging.getLogger(__name__)
logger.info("Logging initialised. Log file → %s", LOG_FILE)

## Cell 4 — Helper Functions

In [ ]:
def clean_cell(value) -> str:
    """Convert a cell value to a stripped string; return empty string if None."""
    if value is None:
        return ""
    return str(value).strip()


def sanitize_filename(name: str) -> str:
    """
    Remove / replace characters that are illegal in Windows / Linux filenames.
    Truncate to 180 chars to avoid path-length issues.
    """
    name = re.sub(r'[\\/:*?"<>|]', "_", name)   # replace illegal chars
    name = re.sub(r'\s+', " ", name).strip()       # collapse whitespace
    name = name[:180]                               # truncate
    return name or "UNKNOWN_FIELD"


def find_target_sheet(wb: openpyxl.Workbook, candidates: list) -> openpyxl.worksheet.worksheet.Worksheet | None:
    """
    Return the first worksheet whose name matches (case-insensitive)
    one of the candidate names. Falls back to the first sheet if none match.
    """
    sheet_names_lower = {s.lower(): s for s in wb.sheetnames}
    for candidate in candidates:
        if candidate.lower() in sheet_names_lower:
            return wb[sheet_names_lower[candidate.lower()]]
    # Fallback: return first sheet with a warning
    logger.warning("None of %s matched. Using first sheet: '%s'", candidates, wb.sheetnames[0])
    return wb[wb.sheetnames[0]]


def extract_records_from_file(filepath: str) -> list[dict]:
    """
    Open one Excel reconciliation template and extract rows ROW_START–ROW_END.
    Returns a list of dicts (one per non-blank reconciliation row).
    Raises on unrecoverable errors so the caller can log and skip.
    """
    source_name = Path(filepath).name

    # Open workbook in read-only + data-only mode for speed
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)

    ws = find_target_sheet(wb, TARGET_SHEET_NAMES)
    if ws is None:
        raise ValueError(f"No usable sheet found in {source_name}")

    records = []

    for row_num in range(ROW_START, ROW_END + 1):
        recon_element  = clean_cell(ws.cell(row=row_num, column=COL_RECON_ELEMENT).value)
        original_data  = clean_cell(ws.cell(row=row_num, column=COL_ORIGINAL_DATA).value)
        discrepancies  = clean_cell(ws.cell(row=row_num, column=COL_DISCREPANCIES).value)
        evidence       = clean_cell(ws.cell(row=row_num, column=COL_EVIDENCE).value)

        # Skip rows where the reconciliation element is blank
        if not recon_element:
            continue

        records.append({
            "Source_File"           : source_name,
            "Reconciliation_Element": recon_element,
            "Original_Data"         : original_data,
            "Discrepancies_Details" : discrepancies,
            "Evidence"              : evidence,
        })

    wb.close()
    return records


print("✅ Helper functions defined.")

## Cell 5 — Discover All Excel Files

In [ ]:
# Recursively find all .xlsx and .xls files in the input folder
# (excluding temp/lock files that start with ~$)
excel_files = [
    str(p)
    for p in Path(INPUT_FOLDER).rglob("*")
    if p.suffix.lower() in {".xlsx", ".xls"}
    and not p.name.startswith("~$")
]

excel_files.sort()   # consistent ordering

print(f"📂 Input folder : {INPUT_FOLDER}")
print(f"📄 Excel files found : {len(excel_files)}")

if not excel_files:
    print("⚠️  No Excel files found. Check INPUT_FOLDER path.")
else:
    print("\nFirst 5 files:")
    for f in excel_files[:5]:
        print(" ", f)

## Cell 6 — Extract Data from All Files

In [ ]:
all_records   = []   # accumulates every extracted row
failed_files  = []   # files that could not be processed
processed     = 0

total = len(excel_files)

for idx, filepath in enumerate(excel_files, start=1):
    filename = Path(filepath).name
    try:
        records = extract_records_from_file(filepath)
        all_records.extend(records)
        processed += 1
        logger.info("[%d/%d] ✓ %s  →  %d rows", idx, total, filename, len(records))

    except Exception as exc:
        failed_files.append({"File": filename, "Error": str(exc)})
        logger.error("[%d/%d] ✗ %s  →  %s", idx, total, filename, exc)
        logger.debug(traceback.format_exc())

print(f"\n{'='*55}")
print(f"  Files found      : {total}")
print(f"  Files processed  : {processed}")
print(f"  Files failed     : {len(failed_files)}")
print(f"  Total records    : {len(all_records)}")
print(f"{'='*55}")

## Cell 7 — Build Master DataFrame & Preview

In [ ]:
COLUMNS = [
    "Source_File",
    "Reconciliation_Element",
    "Original_Data",
    "Discrepancies_Details",
    "Evidence",
]

if all_records:
    master_df = pd.DataFrame(all_records, columns=COLUMNS)
else:
    master_df = pd.DataFrame(columns=COLUMNS)
    print("⚠️  No records extracted. Master DataFrame is empty.")

print(f"Master DataFrame shape: {master_df.shape}")
print(f"Unique source files   : {master_df['Source_File'].nunique()}")
print(f"Unique recon elements : {master_df['Reconciliation_Element'].nunique()}")
print()
master_df.head(10)

## Cell 8 — Export Consolidated Outputs (Excel + CSV)

In [ ]:
# ── Consolidated Excel ──────────────────────────────────────────
consolidated_xlsx = os.path.join(OUTPUT_FOLDER, "consolidated_output.xlsx")

with pd.ExcelWriter(consolidated_xlsx, engine="openpyxl") as writer:
    master_df.to_excel(writer, index=False, sheet_name="All_Records")

    # Auto-size columns for readability
    ws = writer.sheets["All_Records"]
    for col_cells in ws.columns:
        max_len = max((len(str(c.value)) if c.value else 0) for c in col_cells)
        ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 4, 80)

print(f"✅ Consolidated Excel saved → {consolidated_xlsx}")

# ── Consolidated CSV ────────────────────────────────────────────
consolidated_csv = os.path.join(OUTPUT_FOLDER, "consolidated_output.csv")
master_df.to_csv(consolidated_csv, index=False, encoding="utf-8-sig")
print(f"✅ Consolidated CSV   saved → {consolidated_csv}")

## Cell 9 — Generate Field-Level Excel Files (~35 files)

In [ ]:
field_files_created = 0
field_files_failed  = 0

if master_df.empty:
    print("⚠️  No data to split into field-level files.")
else:
    # Group by Reconciliation_Element
    grouped = master_df.groupby("Reconciliation_Element", sort=False)
    unique_fields = master_df["Reconciliation_Element"].unique()
    print(f"Found {len(unique_fields)} unique reconciliation fields.\n")

    # Track sanitized names to detect collisions
    seen_names: dict[str, int] = {}

    for field_name, group_df in grouped:
        safe_name = sanitize_filename(str(field_name))

        # Handle duplicate sanitized names by appending a counter
        if safe_name in seen_names:
            seen_names[safe_name] += 1
            safe_name = f"{safe_name}_{seen_names[safe_name]}"
        else:
            seen_names[safe_name] = 0

        out_path = os.path.join(FIELD_OUTPUT_FOLDER, f"{safe_name}.xlsx")

        try:
            with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
                group_df.reset_index(drop=True).to_excel(
                    writer, index=False, sheet_name="Data"
                )
                # Auto-size columns
                ws = writer.sheets["Data"]
                for col_cells in ws.columns:
                    max_len = max((len(str(c.value)) if c.value else 0) for c in col_cells)
                    ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 4, 80)

            field_files_created += 1
            logger.info("Field file saved: %s  (%d rows)", safe_name + ".xlsx", len(group_df))

        except Exception as exc:
            field_files_failed += 1
            logger.error("Failed to save field file '%s': %s", safe_name, exc)

    print(f"✅ Field-level files created : {field_files_created}")
    print(f"❌ Field-level files failed  : {field_files_failed}")
    print(f"📁 Saved to → {FIELD_OUTPUT_FOLDER}")

## Cell 10 — Failed Files Report

In [ ]:
if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_path = os.path.join(OUTPUT_FOLDER, "failed_files.xlsx")
    failed_df.to_excel(failed_path, index=False)
    print(f"⚠️  {len(failed_files)} file(s) failed. Report saved → {failed_path}")
    print()
    print(failed_df.to_string(index=False))
else:
    print("✅ No failed files — all templates processed successfully!")

## Cell 11 — Final Processing Summary

In [ ]:
print("="*60)
print("          RECONCILIATION EXTRACTION — SUMMARY")
print("="*60)
print(f"  📂  Total Excel files found      : {total}")
print(f"  ✅  Files processed successfully : {processed}")
print(f"  ❌  Files failed / skipped       : {len(failed_files)}")
print(f"  📝  Total records extracted      : {len(master_df)}")
print(f"  🏷️   Unique reconciliation fields : {master_df['Reconciliation_Element'].nunique() if not master_df.empty else 0}")
print(f"  📊  Field-level Excel files      : {field_files_created}")
print()
print("  OUTPUT FILES:")
print(f"    • {consolidated_xlsx}")
print(f"    • {consolidated_csv}")
print(f"    • {FIELD_OUTPUT_FOLDER}/  ({field_files_created} files)")
if failed_files:
    print(f"    • {os.path.join(OUTPUT_FOLDER, 'failed_files.xlsx')}")
print(f"    • {LOG_FILE}")
print("="*60)

## Cell 12 — (Optional) Quick Validation Preview

In [ ]:
# Show how many records each reconciliation field has across all models
if not master_df.empty:
    field_summary = (
        master_df.groupby("Reconciliation_Element")
        .agg(
            File_Count=("Source_File", "nunique"),
            Record_Count=("Source_File", "count"),
            With_Discrepancies=("Discrepancies_Details", lambda x: (x != "").sum()),
        )
        .reset_index()
        .sort_values("Record_Count", ascending=False)
    )
    print("Field-level coverage summary:")
    display(field_summary)

    # Save summary
    summary_path = os.path.join(OUTPUT_FOLDER, "field_summary.xlsx")
    field_summary.to_excel(summary_path, index=False)
    print(f"\n✅ Summary saved → {summary_path}")
else:
    print("No data available for summary.")